# mshook notes (HF rewrite, self-contained)

From: <https://github.com/wjessup/simpleGPT2>

- <https://claude.ai/chat/193c13b8-aee5-40e7-a87b-1c54fa7446a0>

Fully self-contained version -- no `git clone` of the original repo is
needed. Both the BPE `Encoder` (originally `encoder.py`) and the weight
loader (originally `utils.py`) are defined directly in this notebook, and
weights/tokenizer files come from **Hugging Face** instead of OpenAI's
blob storage, read via a tiny hand-rolled **safetensors** parser -- no
TensorFlow dependency.

Requires network access to `huggingface.co` (add it under Settings ->
Capabilities -> Code execution and file creation -> Additional allowed
domains, if running this in a sandboxed Claude environment).

In [1]:
! pip install -q numpy requests tqdm regex


## BPE encoder/decoder (inline, replaces encoder.py)

In [2]:
"""Byte pair encoding utilities.

Adapted from: https://github.com/openai/gpt-2/blob/master/src/encoder.py
(via https://github.com/wjessup/simpleGPT2). Only the encoder/decoder
plumbing is kept here -- the local-file `get_encoder()` loader is dropped
since this notebook loads vocab/merges from Hugging Face instead.
"""
from functools import lru_cache

import regex as re


@lru_cache()
def bytes_to_unicode():
    """
    Returns list of utf-8 byte and a corresponding list of unicode strings.
    The reversible bpe codes work on unicode strings.
    This means you need a large # of unicode characters in your vocab if you want to avoid UNKs.
    When you're at something like a 10B token dataset you end up needing around 5K for decent coverage.
    This is a significant percentage of your normal, say, 32K bpe vocab.
    To avoid that, we want lookup tables between utf-8 bytes and unicode strings.
    And avoids mapping to whitespace/control characters the bpe code barfs on.
    """
    bs = list(range(ord("!"), ord("~") + 1)) + list(range(ord("\xa1"), ord("\xac") + 1)) + list(range(ord("\xae"), ord("\xff") + 1))
    cs = bs[:]
    n = 0
    for b in range(2**8):
        if b not in bs:
            bs.append(b)
            cs.append(2**8 + n)
            n += 1
    cs = [chr(n) for n in cs]
    return dict(zip(bs, cs))


def get_pairs(word):
    """Return set of symbol pairs in a word.
    Word is represented as tuple of symbols (symbols being variable-length strings).
    """
    pairs = set()
    prev_char = word[0]
    for char in word[1:]:
        pairs.add((prev_char, char))
        prev_char = char
    return pairs


class Encoder:
    def __init__(self, encoder, bpe_merges, errors="replace"):
        self.encoder = encoder
        self.decoder = {v: k for k, v in self.encoder.items()}
        self.errors = errors  # how to handle errors in decoding
        self.byte_encoder = bytes_to_unicode()
        self.byte_decoder = {v: k for k, v in self.byte_encoder.items()}
        self.bpe_ranks = dict(zip(bpe_merges, range(len(bpe_merges))))
        self.cache = {}

        # Should have added re.IGNORECASE so BPE merges can happen for capitalized versions of contractions
        self.pat = re.compile(r"""'s|'t|'re|'ve|'m|'ll|'d| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+""")

    def bpe(self, token):
        if token in self.cache:
            return self.cache[token]
        word = tuple(token)
        pairs = get_pairs(word)

        if not pairs:
            return token

        while True:
            bigram = min(pairs, key=lambda pair: self.bpe_ranks.get(pair, float("inf")))
            if bigram not in self.bpe_ranks:
                break
            first, second = bigram
            new_word = []
            i = 0
            while i < len(word):
                try:
                    j = word.index(first, i)
                    new_word.extend(word[i:j])
                    i = j
                except:
                    new_word.extend(word[i:])
                    break

                if word[i] == first and i < len(word) - 1 and word[i + 1] == second:
                    new_word.append(first + second)
                    i += 2
                else:
                    new_word.append(word[i])
                    i += 1
            new_word = tuple(new_word)
            word = new_word
            if len(word) == 1:
                break
            else:
                pairs = get_pairs(word)
        word = " ".join(word)
        self.cache[token] = word
        return word

    def encode(self, text):
        bpe_tokens = []
        for token in re.findall(self.pat, text):
            token = "".join(self.byte_encoder[b] for b in token.encode("utf-8"))
            bpe_tokens.extend(self.encoder[bpe_token] for bpe_token in self.bpe(token).split(" "))
        return bpe_tokens

    def decode(self, tokens):
        text = "".join([self.decoder[token] for token in tokens])
        text = bytearray([self.byte_decoder[c] for c in text]).decode("utf-8", errors=self.errors)
        return text


## HF weight loader (inline, replaces utils.py)

`model_size` maps to a Hugging Face repo:

| model_size | HF repo |
|---|---|
| 124M | gpt2 |
| 355M | gpt2-medium |
| 774M | gpt2-large |
| 1558M | gpt2-xl |

In [3]:
"""HF-based weight loader for simpleGPT2 (numpy-only, no TensorFlow).

Drop-in replacement for the original utils.py. Downloads the tokenizer
and weights from Hugging Face instead of OpenAI's blob storage, and reads
the safetensors weight file with a small hand-rolled parser (no
`safetensors` package required -- just numpy + struct + json).

HF's GPT2 uses Conv1D layers whose weight is stored as [in, out], which
already matches this repo's `linear(x, w, b) = x @ w + b` convention, so
no transposition is needed -- only renaming from HF's flat
`transformer.h.{i}....` keys into this repo's nested params dict.
"""
import json
import os
import struct

import numpy as np
import requests
from tqdm import tqdm

HF_MODEL_NAMES = {
    "124M": "gpt2",
    "355M": "gpt2-medium",
    "774M": "gpt2-large",
    "1558M": "gpt2-xl",
}

HF_BASE = "https://huggingface.co"

_SAFETENSORS_DTYPES = {
    "F64": np.float64,
    "F32": np.float32,
    "F16": np.float16,
    "I64": np.int64,
    "I32": np.int32,
    "I16": np.int16,
    "I8": np.int8,
    "U8": np.uint8,
    "BOOL": np.bool_,
}


def _download(url, dest):
    r = requests.get(url, stream=True)
    r.raise_for_status()
    total = int(r.headers.get("content-length", 0))
    with open(dest, "wb") as f:
        with tqdm(
            ncols=100,
            desc="Fetching " + os.path.basename(dest),
            total=total,
            unit_scale=True,
            unit="b",
        ) as pbar:
            for chunk in r.iter_content(chunk_size=8192):
                f.write(chunk)
                pbar.update(len(chunk))


def download_hf_files(hf_name, model_dir):
    for filename in ["vocab.json", "merges.txt", "config.json", "model.safetensors"]:
        url = f"{HF_BASE}/{hf_name}/resolve/main/{filename}"
        _download(url, os.path.join(model_dir, filename))


def load_safetensors(path):
    """Minimal safetensors reader.

    Format: 8-byte little-endian uint64 header length, then that many
    bytes of JSON header (tensor name -> dtype/shape/data_offsets), then
    the raw tensor bytes.
    """
    with open(path, "rb") as f:
        header_len = struct.unpack("<Q", f.read(8))[0]
        header = json.loads(f.read(header_len))
        data = f.read()

    tensors = {}
    for name, meta in header.items():
        if name == "__metadata__":
            continue
        dtype = _SAFETENSORS_DTYPES[meta["dtype"]]
        start, end = meta["data_offsets"]
        arr = np.frombuffer(data[start:end], dtype=dtype).reshape(meta["shape"])
        tensors[name] = arr.astype(np.float32) if dtype != np.float32 else arr
    return tensors


def get_hf_encoder(model_dir):
    with open(os.path.join(model_dir, "vocab.json"), "r") as f:
        vocab = json.load(f)
    with open(os.path.join(model_dir, "merges.txt"), "r", encoding="utf-8") as f:
        bpe_data = f.read()
    # merges.txt has a leading "#version: ..." comment line, same layout
    # as OpenAI's vocab.bpe -- so the same slicing/split logic applies.
    bpe_merges = [tuple(m.split()) for m in bpe_data.split("\n")[1:-1] if m]
    return Encoder(encoder=vocab, bpe_merges=bpe_merges)


def build_params(tensors, n_layer):
    """Remap the safetensors tensor names into this repo's nested dict shape.

    HF checkpoints for GPT-2 are usually saved with a "transformer."
    prefix (transformer.h.0.ln_1.weight, ...), but some re-uploads or
    conversions drop that prefix and store bare h.0.ln_1.weight keys
    instead. Rather than hard-coding one naming scheme, detect whichever
    prefix is actually present by matching on the tensor name's suffix.
    """
    all_keys = list(tensors.keys())

    def find_key(suffix):
        matches = [k for k in all_keys if k.endswith(suffix)]
        if not matches:
            sample = ", ".join(all_keys[:10])
            raise KeyError(
                f"No tensor found ending in '{suffix}'. "
                f"First tensor names actually in the checkpoint: {sample} ..."
            )
        if len(matches) > 1:
            raise KeyError(f"Ambiguous match for suffix '{suffix}': {matches}")
        return matches[0]

    def g(suffix):
        return tensors[find_key(suffix)]

    blocks = []
    for i in range(n_layer):
        s = f"h.{i}."
        blocks.append(
            {
                "ln_1": {"g": g(s + "ln_1.weight"), "b": g(s + "ln_1.bias")},
                "attn": {
                    "c_attn": {"w": g(s + "attn.c_attn.weight"), "b": g(s + "attn.c_attn.bias")},
                    "c_proj": {"w": g(s + "attn.c_proj.weight"), "b": g(s + "attn.c_proj.bias")},
                },
                "ln_2": {"g": g(s + "ln_2.weight"), "b": g(s + "ln_2.bias")},
                "mlp": {
                    "c_fc": {"w": g(s + "mlp.c_fc.weight"), "b": g(s + "mlp.c_fc.bias")},
                    "c_proj": {"w": g(s + "mlp.c_proj.weight"), "b": g(s + "mlp.c_proj.bias")},
                },
            }
        )

    return {
        "wte": g("wte.weight"),
        "wpe": g("wpe.weight"),
        "blocks": blocks,
        "ln_f": {"g": g("ln_f.weight"), "b": g("ln_f.bias")},
    }


def load_encoder_hparams_and_params(model_size, models_dir):
    assert model_size in HF_MODEL_NAMES, f"model_size must be one of {list(HF_MODEL_NAMES)}"
    hf_name = HF_MODEL_NAMES[model_size]

    model_dir = os.path.join(models_dir, model_size)
    os.makedirs(model_dir, exist_ok=True)

    weights_path = os.path.join(model_dir, "model.safetensors")
    # A real GPT-2 (124M) safetensors file is >400MB; anything much smaller
    # means a previous download was interrupted or hit an error page --
    # redownload rather than silently parsing a truncated/bad file.
    if not os.path.exists(weights_path) or os.path.getsize(weights_path) < 10_000_000:
        download_hf_files(hf_name, model_dir)

    with open(os.path.join(model_dir, "config.json")) as f:
        config = json.load(f)

    hparams = {
        "n_vocab": config["vocab_size"],
        "n_ctx": config.get("n_ctx", config["n_positions"]),
        "n_embd": config["n_embd"],
        "n_head": config["n_head"],
        "n_layer": config["n_layer"],
    }

    encoder = get_hf_encoder(model_dir)
    tensors = load_safetensors(weights_path)
    params = build_params(tensors, hparams["n_layer"])

    return encoder, hparams, params


In [4]:
import numpy as np
model_size: str = "124M"
#model_size: str = "355M"
#model_size: str = "774M"
#model_size: str = "1558M"
models_dir: str = "models"
encoder, hparams, params = load_encoder_hparams_and_params(model_size, models_dir)

def gelu(x):
    return 0.5 * x * (1 + np.tanh(np.sqrt(2 / np.pi) * (x + 0.044715 * x**3)))

def softmax(x):
    exp_x = np.exp(x - np.max(x, axis=-1, keepdims=True))
    return exp_x / np.sum(exp_x, axis=-1, keepdims=True)

def layer_norm(x, g, b, eps: float = 1e-5):
    mean = np.mean(x, axis=-1, keepdims=True)
    variance = np.var(x, axis=-1, keepdims=True)
    return g * (x - mean) / np.sqrt(variance + eps) + b

def linear(x, w, b):
    return x @ w + b

def ffn(x, c_fc, c_proj):
    return linear(gelu(linear(x, **c_fc)), **c_proj)

def attention(q, k, v, mask):
    attention_scores = softmax(q @ k.T / np.sqrt(q.shape[-1]) + mask)
    return attention_scores @ v


Fetching vocab.json: 1.04Mb [00:00, 20.7Mb/s]
Fetching merges.txt: 456kb [00:00, 2.01Mb/s]
Fetching config.json: 100%|████████████████████████████████████████| 665/665 [00:00<00:00, 2.68Mb/s]
Fetching model.safetensors: 100%|████████████████████████████████| 548M/548M [00:09<00:00, 57.7Mb/s]


In [5]:
def main(prompt: str, n_tokens_to_generate: int = 10):
    inputs = encoder.encode(prompt)
    assert len(inputs) + n_tokens_to_generate < hparams["n_ctx"]

    for _ in range(n_tokens_to_generate):

        x = params['wte'][inputs] + params['wpe'][range(len(inputs))]
        causal_mask = (1 - np.tri(x.shape[0], dtype=x.dtype)) * -1e10

        for block in params['blocks']:

            # layer norm 1
            ln1 = layer_norm(x, **block['ln_1']) # (seq, 768) => (6, 768)

            # attention
            qkv = linear(ln1, **block['attn']['c_attn']) # => (6, 2304)

            qkv_heads = np.split(qkv, 3*hparams['n_head'], axis=-1)

            attn_out = []
            for head_id in range(0, hparams['n_head']):
                out = attention(qkv_heads[head_id],
                                qkv_heads[head_id + hparams['n_head']],
                                qkv_heads[head_id + hparams['n_head']*2],
                                causal_mask)
                attn_out.append(out)


            attn = linear(np.hstack(attn_out), **block['attn']['c_proj'])

            # residual stream
            x = x + attn

            # layer norm 2
            ln2 = layer_norm(x, **block['ln_2'])

            # feed forward (or MLP)
            ffn_out = ffn(ln2, **block['mlp'])

            # residual stream
            x = x + ffn_out

        logits =  layer_norm(x[-1], **params['ln_f']) @ params['wte'].T

        next_id = np.argmax(logits)
        #print(x.shape)
        #print(logits.shape)  #
        print(encoder.decode([int(next_id)]), end="", flush=True)
        inputs.append(int(next_id))
    output_ids =  inputs[len(inputs) - n_tokens_to_generate :]

    output_text = encoder.decode(output_ids)
    #return output_text

(50257,) is a 1-D vector — one score (logit) per possible next token, since GPT-2's BPE vocabulary has exactly 50,257 tokens.

In the code, logits = layer_norm(x[-1], **params['ln_f']) @ params['wte'].T:

x[-1] is just the last position's residual-stream vector, shape (n_embd,) — e.g. (768,) for 124M.
params['wte'].T has shape (n_embd, n_vocab) = (768, 50257).
The matmul projects that single embedding onto every row of the (transposed, tied) embedding matrix, giving one unnormalized score per vocab entry: shape (50257,).

np.argmax(logits) then picks the index of the highest-scoring token, and encoder.decode([...]) turns that token id back into text. If you softmax() it first you'd get a proper probability distribution over the vocab instead of raw scores — same shape, but summing to 1.

In [6]:

if __name__ == "__main__":
    prompt = "not all heroes wear capes except"
    #prompt = "Four score and seven"
    #prompt = "never have I"
    prompt = "How strange the change from minor"
    main(prompt)
    #print("\n all done! \n output: " + output)

 to major is.

The first thing to

In [ ]:
while True:
    prompt = input("Prompt (blank to quit): ")
    if not prompt:
        break
    n_tokens_to_generate = input("Tokens to generate [10]: ").strip()
    n_tokens_to_generate = int(n_tokens_to_generate) if n_tokens_to_generate else 10

    main(prompt, n_tokens_to_generate)
    print()  # newline after the streamed tokens, before the next prompt

Prompt (blank to quit): I come to bury Caeser not
Tokens to generate [10]: 
 because I am a man, but because I am
Prompt (blank to quit): I come to bury Caeser not to praise
Tokens to generate [10]: 
 him, but to praise him who is the Lord


In [ ]:
if __name__ == "__main__":
    prompt = "not all heroes wear capes except"
    prompt = "never have I"
    prompt = "For ever and ever"
    prompt = "When there's a chance I will"
    #prompt = "a a b a a b a a"
    prompt = "Paris is the capital of"
    #prompt = "The capital of France is"
    prompt = "A short poem is"
    prompt = "When Mary and John went to the store, John gave a drink to"
    prompt = "Beats Music is owned by"
    prompt = "1.2.8.+3.2.7.+1.2.8.+3.2.7.+1.2.8.+3.2."
    prompt = "3 7 8 2 3 7 8"
    prompt = "not all heroes wear capes except"
    prompt = "Paris is the capital of"
    prompt = "never have I"
    prompt = "When there's a chance I will"
    prompt = "A short poem is"
    prompt = "When Mary and John went to the store, John gave a drink to"
    prompt = "Beats Music is owned by"
    main(prompt)
    #print("\n all done! \n output: " + output)
